# DocuDiff AI — Complete Google Colab Implementation
### T4 GPU • Qwen2.5-0.5B-Instruct • Sentence Transformers • FAISS • LangGraph

This notebook is designed to be run top-to-bottom in Google Colab. It implements an end-to-end document comparison workflow:

1. Install and verify dependencies / GPU.
2. Upload two versions of a PDF, DOCX, or TXT document (or use the included sample files).
3. Extract text while preserving page/paragraph references.
4. Chunk text and create sentence embeddings.
5. Align passages with exact matching + FAISS semantic retrieval.
6. Detect numerical, date, negation, obligation, and condition changes.
7. Use Qwen2.5-0.5B-Instruct to explain candidate changes with validated JSON output.
8. Orchestrate analysis with LangGraph (with a transparent fallback if graph setup fails).
9. Visualize results and generate JSON, CSV, HTML, charts, and a ZIP bundle for download.

**Important:** Qwen's explanations are AI-assisted and may be wrong. The deterministic comparison and rule flags are evidence aids, not legal advice. Verify important changes against the original documents.

**Recommended runtime:** Colab → Runtime → Change runtime type → T4 GPU. Run every cell in order.

## 0. Runtime and Google Drive (optional persistence)

In [ ]:
import os, sys, platform, json, re, time, uuid, shutil, zipfile, textwrap, html, csv, math, warnings
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore")
print("Python:", sys.version)
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("PyTorch not ready yet:", e)

# Optional: mount Google Drive to persist model cache and outputs across Colab sessions.
MOUNT_DRIVE = False  # Change to True if you want Drive persistence.
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ["HF_HOME"] = "/content/drive/MyDrive/.cache/huggingface"

## 1. Install required packages

In [ ]:
# Colab usually provides PyTorch already. Install the project libraries.
%pip -q install -U transformers accelerate sentence-transformers faiss-cpu pymupdf python-docx langgraph pydantic pandas matplotlib seaborn

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import faiss
import fitz  # PyMuPDF
from docx import Document
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, Markdown, HTML
from google.colab import files

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Embeddings can run on CPU; Qwen inference may be slow.")

## 2. Configuration and sample documents

Set the model IDs and thresholds here. Similarity thresholds are starting values and should be tuned using your labelled examples. The included sample policy is only a demo fixture.

In [ ]:
GENERATION_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EXACT_THRESHOLD = 0.999
SEMANTIC_MATCH_THRESHOLD = 0.88
CANDIDATE_THRESHOLD = 0.60
TOP_K = 3
MAX_NEW_TOKENS = 220
MAX_UPLOAD_MB = 15
OUTPUT_DIR = Path("/content/docudiff_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OLD_SAMPLE = """Employee Leave Policy
Employees may carry forward up to 15 days of unused leave.
A leave request must be submitted at least 7 days before planned travel.
The system should retain access logs for 30 days.
Access is permitted for approved administrators only.
Unused medical leave is not transferable.
"""

NEW_SAMPLE = """Employee Leave Policy
Employees may carry forward up to 10 days of unused leave.
A leave request must be submitted at least 5 days before planned travel.
The system must retain access logs for 30 days.
Access is permitted for approved administrators and auditors.
Unused medical leave is not transferable.
Emergency access requests require manager approval.
"""

print("Configuration ready.")

## 3. Upload documents or use the built-in sample

In [ ]:
USE_SAMPLE_FILES = True  # Set False to upload your own two documents.
uploaded_old_path = None
uploaded_new_path = None

if USE_SAMPLE_FILES:
    old_path = Path("/content/policy_old.txt")
    new_path = Path("/content/policy_new.txt")
    old_path.write_text(OLD_SAMPLE, encoding="utf-8")
    new_path.write_text(NEW_SAMPLE, encoding="utf-8")
else:
    print("Upload the OLD version first:")
    up_old = files.upload()
    if not up_old:
        raise ValueError("No old document uploaded.")
    old_name = next(iter(up_old))
    old_path = Path("/content") / old_name
    old_path.write_bytes(up_old[old_name])

    print("Upload the NEW version:")
    up_new = files.upload()
    if not up_new:
        raise ValueError("No new document uploaded.")
    new_name = next(iter(up_new))
    new_path = Path("/content") / new_name
    new_path.write_bytes(up_new[new_name])

for p in [old_path, new_path]:
    if p.stat().st_size > MAX_UPLOAD_MB * 1024 * 1024:
        raise ValueError(f"{p.name} exceeds {MAX_UPLOAD_MB} MB.")
    if p.suffix.lower() not in [".pdf", ".docx", ".txt"]:
        raise ValueError(f"Unsupported file type: {p.suffix}")
    print(f"{p.name}: {p.stat().st_size:,} bytes")

## 4. Document extraction with source metadata

In [ ]:
def extract_document(path: Path):
    """Return paragraph-like records with source location metadata."""
    ext = path.suffix.lower()
    records = []
    if ext == ".pdf":
        pdf = fitz.open(path)
        for page_idx, page in enumerate(pdf, start=1):
            text = page.get_text("text") or ""
            for para_idx, part in enumerate([x.strip() for x in re.split(r"\\n\\s*\\n", text) if x.strip()], start=1):
                records.append({"text": part, "page": page_idx, "paragraph": para_idx, "source": path.name})
        if not records:
            raise ValueError(f"No extractable text in {path.name}. It may be a scanned PDF; OCR is not included in this MVP.")
    elif ext == ".docx":
        docx = Document(path)
        for idx, paragraph in enumerate(docx.paragraphs, start=1):
            txt = paragraph.text.strip()
            if txt:
                records.append({"text": txt, "page": None, "paragraph": idx, "source": path.name})
        # Preserve table content as rows rather than silently dropping it.
        table_idx = 0
        for ti, table in enumerate(docx.tables, start=1):
            for ri, row in enumerate(table.rows, start=1):
                txt = " | ".join(cell.text.strip() for cell in row.cells if cell.text.strip())
                if txt:
                    records.append({"text": txt, "page": None, "paragraph": None, "source": path.name, "table": ti, "row": ri})
        if not records:
            raise ValueError(f"No text found in {path.name}.")
    elif ext == ".txt":
        raw = path.read_text(encoding="utf-8-sig", errors="replace")
        for idx, part in enumerate([x.strip() for x in re.split(r"\\n\\s*\\n|(?<=[.!?])\\s+(?=[A-Z])", raw) if x.strip()], start=1):
            records.append({"text": part, "page": None, "paragraph": idx, "source": path.name})
        if not records:
            raise ValueError(f"No text found in {path.name}.")
    else:
        raise ValueError(f"Unsupported extension: {ext}")
    return records

old_records = extract_document(old_path)
new_records = extract_document(new_path)
print(f"Old version: {len(old_records)} extracted passages")
print(f"New version: {len(new_records)} extracted passages")
display(pd.DataFrame(old_records).head())

## 5. Paragraph-aware chunking

In [ ]:
def make_chunks(records, prefix, max_chars=900):
    """Combine adjacent short paragraphs while retaining all source references."""
    chunks, buffer, refs, count = [], [], [], 0
    def flush():
        nonlocal buffer, refs, count
        if buffer:
            chunks.append({
                "chunk_id": f"{prefix}_c{len(chunks)+1:04d}",
                "text": "\\n".join(buffer),
                "sources": refs.copy()
            })
            buffer, refs, count = [], [], 0

    for rec in records:
        text_part = rec["text"].strip()
        if not text_part:
            continue
        if len(text_part) > max_chars:
            flush()
            # Split unusually long paragraph while preserving the paragraph reference.
            for start in range(0, len(text_part), max_chars):
                piece = text_part[start:start+max_chars]
                chunks.append({"chunk_id": f"{prefix}_c{len(chunks)+1:04d}",
                               "text": piece, "sources": [{k:v for k,v in rec.items() if k != "text"}]})
            continue
        if count + len(text_part) > max_chars:
            flush()
        buffer.append(text_part)
        refs.append({k:v for k,v in rec.items() if k != "text"})
        count += len(text_part)
    flush()
    return chunks

old_chunks = make_chunks(old_records, "old")
new_chunks = make_chunks(new_records, "new")
print("Old chunks:", len(old_chunks), "| New chunks:", len(new_chunks))

## 6. Load embedding model and build FAISS indices

In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")

def embed_texts(texts):
    if not texts:
        return np.empty((0, 384), dtype="float32")
    arr = embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    return np.asarray(arr, dtype="float32")

old_vectors = embed_texts([c["text"] for c in old_chunks])
new_vectors = embed_texts([c["text"] for c in new_chunks])

def build_faiss(vectors):
    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)  # normalized vectors => inner product is cosine similarity
    if len(vectors):
        index.add(vectors)
    return index

old_index = build_faiss(old_vectors)
new_index = build_faiss(new_vectors)
print("Embedding dimension:", old_vectors.shape[1], "| FAISS old/new:", old_index.ntotal, new_index.ntotal)

## 7. Exact + semantic passage alignment

In [ ]:
def norm_text(s):
    return re.sub(r"\\s+", " ", re.sub(r"[^\\w%$./-]", " ", s.lower())).strip()

def align_chunks(old_chunks, new_chunks, old_vecs, new_vecs, threshold=SEMANTIC_MATCH_THRESHOLD):
    """One-to-one greedy matching. Exact normalized matches first, then semantic similarity."""
    matches, used_old, used_new = [], set(), set()
    old_norm = {}
    for i, ch in enumerate(old_chunks):
        old_norm.setdefault(norm_text(ch["text"]), []).append(i)

    # Exact normalized matches.
    for j, ch in enumerate(new_chunks):
        key = norm_text(ch["text"])
        candidates = [i for i in old_norm.get(key, []) if i not in used_old]
        if candidates:
            i = candidates[0]
            used_old.add(i); used_new.add(j)
            matches.append({"old_i": i, "new_i": j, "similarity": 1.0, "match_kind": "exact"})

    # Semantic candidates from FAISS.
    if len(old_chunks) and len(new_chunks):
        idx = build_faiss(old_vecs)
        sims, inds = idx.search(new_vecs, min(TOP_K, len(old_chunks)))
        candidates = []
        for j in range(len(new_chunks)):
            for sim, i in zip(sims[j], inds[j]):
                if i >= 0 and i not in used_old and j not in used_new:
                    candidates.append((float(sim), int(i), j))
        candidates.sort(reverse=True)
        for sim, i, j in candidates:
            if sim >= CANDIDATE_THRESHOLD and i not in used_old and j not in used_new:
                used_old.add(i); used_new.add(j)
                matches.append({"old_i": i, "new_i": j, "similarity": sim,
                                "match_kind": "semantic" if sim < 0.999 else "near_exact"})
    unmatched_old = [i for i in range(len(old_chunks)) if i not in used_old]
    unmatched_new = [j for j in range(len(new_chunks)) if j not in used_new]
    return matches, unmatched_old, unmatched_new

matches, unmatched_old, unmatched_new = align_chunks(old_chunks, new_chunks, old_vectors, new_vectors)
print("Matched:", len(matches), "Removed candidates:", len(unmatched_old), "Added candidates:", len(unmatched_new))

## 8. Deterministic rule checks

In [ ]:
NUMBER_RE = re.compile(r"\\b\\d+(?:[.,]\\d+)?\\s*(?:%|days?|hours?|weeks?|months?|years?|km|kg|mb|gb)?\\b", re.I)
DATE_RE = re.compile(r"\\b(?:\\d{1,2}[/-]\\d{1,2}[/-]\\d{2,4}|\\d{4}-\\d{2}-\\d{2}|Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\\b", re.I)
NEGATION_RE = re.compile(r"\\b(not|no|never|without|except|cannot|can't|prohibited|exclude|excluding)\\b", re.I)
MODAL_RE = re.compile(r"\\b(may|can|could|should|shall|must|required|optional|prohibited|permitted|allowed)\\b", re.I)
CONDITION_RE = re.compile(r"\\b(if|unless|only when|provided that|except when|subject to|in the event)\\b", re.I)

def rule_flags(old_text, new_text):
    flags = []
    old_nums, new_nums = NUMBER_RE.findall(old_text), NUMBER_RE.findall(new_text)
    if old_nums != new_nums:
        flags.append(f"numeric_or_unit_change: {old_nums} -> {new_nums}")
    old_dates, new_dates = DATE_RE.findall(old_text), DATE_RE.findall(new_text)
    if old_dates != new_dates:
        flags.append(f"date_change: {old_dates} -> {new_dates}")
    if NEGATION_RE.findall(old_text.lower()) != NEGATION_RE.findall(new_text.lower()):
        flags.append("negation_or_exclusion_language_changed")
    if MODAL_RE.findall(old_text.lower()) != MODAL_RE.findall(new_text.lower()):
        flags.append("permission_or_obligation_language_changed")
    if CONDITION_RE.findall(old_text.lower()) != CONDITION_RE.findall(new_text.lower()):
        flags.append("condition_or_exception_language_changed")
    return flags

for m in matches[:5]:
    print(rule_flags(old_chunks[m["old_i"]]["text"], new_chunks[m["new_i"]]["text"]))

## 9. Load Qwen2.5-0.5B-Instruct on T4

In [ ]:
# Load once. On a Colab T4, float16 is typically the appropriate dtype.
# If model download fails, continue with LLM_ENABLED=False; deterministic comparison still works.
LLM_ENABLED = False
tokenizer = None
llm = None
try:
    model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL, trust_remote_code=False)
    llm = AutoModelForCausalLM.from_pretrained(
        GENERATION_MODEL,
        torch_dtype=model_dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    LLM_ENABLED = True
    print("Loaded:", GENERATION_MODEL)
    if torch.cuda.is_available():
        print("Allocated GPU GB:", round(torch.cuda.memory_allocated()/1024**3, 2))
except Exception as e:
    print("Qwen loading failed; continuing with deterministic fallback.")
    print(type(e).__name__, str(e)[:500])

## 10. LLM analysis with strict JSON validation

In [ ]:
ALLOWED_CLASSES = {"wording_only", "meaning_changed", "numerical_or_conditional", "uncertain"}
ALLOWED_PRIORITIES = {"low", "medium", "high"}

def safe_fallback(flags):
    if flags:
        cls, priority = "numerical_or_conditional", "high" if any("negation" in f or "obligation" in f or "numeric" in f for f in flags) else "medium"
        explanation = "Deterministic checks found potentially material token changes. Review the paired excerpts."
    else:
        cls, priority = "uncertain", "medium"
        explanation = "The passages differ, but automated analysis could not confidently determine whether the meaning changed."
    return {"semantic_class": cls, "priority": priority, "confidence": 0.35,
            "explanation": explanation, "implication": "Human review is recommended.",
            "requires_human_review": True}

def analyze_pair(old_text, new_text, flags):
    if not LLM_ENABLED:
        return safe_fallback(flags)
    prompt = f'''You are a cautious document-change analyst. Compare only the supplied excerpts.
Do not invent facts or claim legal conclusions. A wording change is not automatically a meaning change.
Pay attention to numbers, dates, thresholds, negation, permissions, obligations, conditions and exceptions.
If evidence is insufficient, choose uncertain. Return one JSON object only.

OLD EXCERPT:
{old_text[:1800]}

NEW EXCERPT:
{new_text[:1800]}

DETERMINISTIC FLAGS:
{flags}

Return exactly this schema:
{{"semantic_class":"wording_only|meaning_changed|numerical_or_conditional|uncertain",
"priority":"low|medium|high","confidence":0.0,"explanation":"brief evidence-based explanation",
"implication":"possible practical effect or no material effect apparent",
"requires_human_review":true}}
'''
    try:
        messages = [{"role":"user", "content":prompt}]
        if hasattr(tokenizer, "apply_chat_template"):
            inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                                    return_tensors="pt", return_dict=True)
        else:
            inputs = tokenizer(prompt, return_tensors="pt")
        inputs = {k:v.to(llm.device) for k,v in inputs.items()}
        with torch.inference_mode():
            output = llm.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                  temperature=None, pad_token_id=tokenizer.pad_token_id)
        generated = output[0][inputs["input_ids"].shape[1]:]
        raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
        # Extract the first JSON object, tolerating code fences.
        match = re.search(r"\\{.*\\}", raw, flags=re.S)
        if not match:
            raise ValueError("No JSON object in model output")
        obj = json.loads(match.group(0))
        if obj.get("semantic_class") not in ALLOWED_CLASSES: raise ValueError("Invalid semantic_class")
        if obj.get("priority") not in ALLOWED_PRIORITIES: raise ValueError("Invalid priority")
        obj["confidence"] = float(max(0, min(1, obj.get("confidence", 0.0))))
        obj["explanation"] = str(obj.get("explanation", ""))[:700]
        obj["implication"] = str(obj.get("implication", ""))[:700]
        obj["requires_human_review"] = bool(obj.get("requires_human_review", True))
        return obj
    except Exception as e:
        result = safe_fallback(flags)
        result["llm_error"] = f"{type(e).__name__}: {str(e)[:200]}"
        return result

## 11. LangGraph orchestration

In [ ]:
# A small, transparent graph: analyze -> validate/fallback -> finalize.
# If LangGraph is unavailable in the runtime, the same function executes sequentially.
try:
    from typing import TypedDict, Any
    from langgraph.graph import StateGraph, END

    class AnalysisState(TypedDict, total=False):
        old_text: str
        new_text: str
        flags: list
        result: dict

    def node_analyze(state: AnalysisState):
        return {"result": analyze_pair(state["old_text"], state["new_text"], state.get("flags", []))}

    def node_validate(state: AnalysisState):
        r = state.get("result", {})
        valid = r.get("semantic_class") in ALLOWED_CLASSES and r.get("priority") in ALLOWED_PRIORITIES
        return {"result": r if valid else safe_fallback(state.get("flags", []))}

    graph_builder = StateGraph(AnalysisState)
    graph_builder.add_node("analyze", node_analyze)
    graph_builder.add_node("validate", node_validate)
    graph_builder.set_entry_point("analyze")
    graph_builder.add_edge("analyze", "validate")
    graph_builder.add_edge("validate", END)
    analysis_graph = graph_builder.compile()

    def run_analysis(old_text, new_text, flags):
        return analysis_graph.invoke({"old_text":old_text, "new_text":new_text, "flags":flags})["result"]
    LANGGRAPH_ENABLED = True
except Exception as e:
    print("LangGraph unavailable; sequential fallback enabled:", str(e)[:250])
    LANGGRAPH_ENABLED = False
    def run_analysis(old_text, new_text, flags):
        return analyze_pair(old_text, new_text, flags)

print("LangGraph enabled:", LANGGRAPH_ENABLED)

## 12. Run complete comparison

In [ ]:
def source_label(chunk):
    if not chunk.get("sources"): return "Source location unavailable"
    s = chunk["sources"][0]
    parts = [s.get("source", "")]
    if s.get("page") is not None: parts.append(f"page {s['page']}")
    if s.get("paragraph") is not None: parts.append(f"paragraph {s['paragraph']}")
    if s.get("table") is not None: parts.append(f"table {s['table']} row {s.get('row')}")
    return ", ".join(x for x in parts if x)

change_records = []
for m in matches:
    oc, nc = old_chunks[m["old_i"]], new_chunks[m["new_i"]]
    if norm_text(oc["text"]) == norm_text(nc["text"]):
        continue
    flags = rule_flags(oc["text"], nc["text"])
    analysis = run_analysis(oc["text"], nc["text"], flags)
    change_records.append({
        "change_type": "modified", "old_text": oc["text"], "new_text": nc["text"],
        "old_source": source_label(oc), "new_source": source_label(nc),
        "similarity": round(m["similarity"], 4), "match_kind": m["match_kind"],
        "rule_flags": flags, **analysis
    })

for i in unmatched_old:
    oc = old_chunks[i]
    change_records.append({
        "change_type":"removed", "old_text":oc["text"], "new_text":"",
        "old_source":source_label(oc), "new_source":"—", "similarity":None,
        "match_kind":"unmatched", "rule_flags":[],
        "semantic_class":"not_applicable", "priority":"medium",
        "confidence":0.7, "explanation":"This passage appears in the old version but no counterpart was confidently matched in the new version.",
        "implication":"Confirm whether the content was intentionally removed.", "requires_human_review":True
    })
for j in unmatched_new:
    nc = new_chunks[j]
    change_records.append({
        "change_type":"added", "old_text":"", "new_text":nc["text"],
        "old_source":"—", "new_source":source_label(nc), "similarity":None,
        "match_kind":"unmatched", "rule_flags":[],
        "semantic_class":"not_applicable", "priority":"medium",
        "confidence":0.7, "explanation":"This passage appears in the new version without a confidently matched passage in the old version.",
        "implication":"Review the newly introduced content.", "requires_human_review":True
    })

summary = {
    "comparison_id": str(uuid.uuid4())[:8],
    "old_filename": old_path.name, "new_filename": new_path.name,
    "created_at": datetime.utcnow().isoformat() + "Z",
    "total_changes": len(change_records),
    "added": sum(x["change_type"]=="added" for x in change_records),
    "removed": sum(x["change_type"]=="removed" for x in change_records),
    "modified": sum(x["change_type"]=="modified" for x in change_records),
    "high_priority": sum(x["priority"]=="high" for x in change_records),
    "model": GENERATION_MODEL if LLM_ENABLED else "deterministic fallback",
    "embedding_model": EMBEDDING_MODEL,
    "langgraph": LANGGRAPH_ENABLED
}
print(json.dumps(summary, indent=2))

## 13. Visual analytics

In [ ]:
if change_records:
    df = pd.DataFrame(change_records)
    display(df[["change_type","semantic_class","priority","similarity","old_source","new_source"]].fillna("—"))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df["change_type"].value_counts().reindex(["added","removed","modified"], fill_value=0).plot(
        kind="bar", ax=axes[0], title="Changes by type")
    df["priority"].value_counts().reindex(["low","medium","high"], fill_value=0).plot(
        kind="bar", ax=axes[1], title="Review priority")
    axes[0].set_xlabel("Change type"); axes[0].set_ylabel("Count")
    axes[1].set_xlabel("Priority"); axes[1].set_ylabel("Count")
    plt.tight_layout()
    chart_path = OUTPUT_DIR / "change_analytics.png"
    fig.savefig(chart_path, dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("No changes detected.")

## 14. Inspect individual changes

In [ ]:
for idx, item in enumerate(change_records, start=1):
    print(f"\\n{'='*90}\\nCHANGE {idx}: {item['change_type'].upper()} | {item['semantic_class']} | PRIORITY {item['priority'].upper()}")
    print("Old source:", item["old_source"])
    print("New source:", item["new_source"])
    if item["old_text"]: print("\\nOLD:\\n", item["old_text"])
    if item["new_text"]: print("\\nNEW:\\n", item["new_text"])
    print("\\nExplanation:", item["explanation"])
    print("Implication:", item["implication"])
    print("Rule flags:", item["rule_flags"])
    print("Similarity:", item["similarity"], "| Confidence:", item["confidence"])

## 15. Export JSON, CSV, HTML, charts, and ZIP

In [ ]:
result_payload = {"summary": summary, "changes": change_records,
                  "warnings": ["Scanned PDFs without extractable text are not OCR-processed.",
                               "Priority is a review triage label, not a legal risk determination.",
                               "Model-estimated confidence is not calibrated."]}
json_path = OUTPUT_DIR / "comparison_result.json"
json_path.write_text(json.dumps(result_payload, indent=2, ensure_ascii=False), encoding="utf-8")

csv_path = OUTPUT_DIR / "comparison_changes.csv"
if change_records:
    pd.DataFrame(change_records).to_csv(csv_path, index=False)
else:
    pd.DataFrame(columns=["change_type","old_text","new_text","priority"]).to_csv(csv_path, index=False)

def html_report(payload):
    s = payload["summary"]
    parts = [f"<h1>DocuDiff AI — Comparison Report</h1><p>{html.escape(s['old_filename'])} → {html.escape(s['new_filename'])}</p>",
             f"<p>Total: {s['total_changes']} | Added: {s['added']} | Removed: {s['removed']} | Modified: {s['modified']} | High priority: {s['high_priority']}</p>"]
    for i, ch in enumerate(payload["changes"], 1):
        parts.append(f"<section><h2>{i}. {html.escape(ch['change_type'])} — {html.escape(ch['semantic_class'])} — {html.escape(ch['priority'])}</h2>")
        parts.append(f"<p><b>Old source:</b> {html.escape(str(ch['old_source']))}<br><b>New source:</b> {html.escape(str(ch['new_source']))}</p>")
        parts.append(f"<h3>Old passage</h3><pre>{html.escape(ch['old_text'])}</pre><h3>New passage</h3><pre>{html.escape(ch['new_text'])}</pre>")
        parts.append(f"<p><b>Explanation:</b> {html.escape(ch['explanation'])}</p><p><b>Implication:</b> {html.escape(ch['implication'])}</p>")
        parts.append(f"<p><b>Rule flags:</b> {html.escape(', '.join(ch['rule_flags']))}</p></section>")
    parts.append("<hr><p>AI-assisted review only. Verify material changes against original documents.</p>")
    return "<html><head><meta charset='utf-8'><style>body{font-family:Arial;max-width:1000px;margin:40px auto;padding:0 20px}section{border-top:1px solid #ddd;padding:16px 0}pre{white-space:pre-wrap;background:#f4f4f4;padding:12px;border-radius:6px}</style></head><body>" + "".join(parts) + "</body></html>"

html_path = OUTPUT_DIR / "comparison_report.html"
html_path.write_text(html_report(result_payload), encoding="utf-8")

# Save a machine-readable benchmark row for later manual ground-truth annotation.
eval_path = OUTPUT_DIR / "evaluation_template.csv"
pd.DataFrame([{
    "case_id": f"case_{i+1:03d}", "old_text": ch["old_text"], "new_text": ch["new_text"],
    "predicted_change_type": ch["change_type"], "predicted_semantic_class": ch["semantic_class"],
    "ground_truth_change_type": "", "ground_truth_semantic_class": "", "review_notes": ""
} for i,ch in enumerate(change_records)]).to_csv(eval_path, index=False)

# Include inputs, results and charts in a single downloadable bundle.
bundle_path = Path("/content/DocuDiff_AI_results_bundle.zip")
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT_DIR.glob("*"):
        if path.is_file():
            z.write(path, arcname=f"results/{path.name}")
    z.write(old_path, arcname=f"inputs/{old_path.name}")
    z.write(new_path, arcname=f"inputs/{new_path.name}")

print("Created:", bundle_path, "| size MB:", round(bundle_path.stat().st_size/1024**2, 2))

## 16. Download your results ZIP

In [ ]:
files.download(str(bundle_path))

## 17. Optional: save the working notebook and output bundle to Google Drive

If Drive was mounted above, uncomment and run this cell. Otherwise, the results ZIP is already downloaded by the previous cell.

In [ ]:
# Uncomment if you mounted Drive:
# destination = Path("/content/drive/MyDrive/DocuDiff_AI")
# destination.mkdir(parents=True, exist_ok=True)
# shutil.copy2(bundle_path, destination / bundle_path.name)
# print("Saved to:", destination)

## 18. Quick evaluation notes

In [ ]:
# A small smoke-check. These are pipeline sanity checks, not a full benchmark score.
assert len(old_records) > 0 and len(new_records) > 0
assert len(old_chunks) > 0 and len(new_chunks) > 0
assert all("text" in c and "sources" in c for c in old_chunks + new_chunks)
assert summary["total_changes"] >= 1, "Sample fixture should contain known changes."
assert json_path.exists() and html_path.exists() and bundle_path.exists()
print("Smoke checks passed.")
print("Do not report precision/recall/F1 until you have manually labelled ground truth.")